# Preprocessing MCTNet — Format **(N, 10, 36)** channels-first

**Format retenu pour le modèle :**
- `Input1` → `(N, C=10, T=36)` — 10 bandes × 36 pas de temps
- `Input2` → `(N, T=36)` — masque manquants
- `Labels` → `(N,)` — entiers

**Normalisation :** `X / 10000` uniquement (conforme paper Section 2.2.4)  
**NDVI :** visualisation uniquement — pas une feature du modèle

## Cellule 1 — Imports

In [ ]:
import numpy as np
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

print('NumPy  :', np.__version__)
print('Pandas :', pd.__version__)

## Cellule 2 — Configuration

In [ ]:
ARK_CSV = 'Arkansas_10k_20m.csv'
CAL_CSV = 'California_10k_20m.csv'
OUT_DIR = 'preprocessed'

BANDS   = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
N_TIMES = 36
N_BANDS = 10
SCALE   = 10000.0

IDX = {band: i for i, band in enumerate(BANDS)}

ARK_LABELS = {0: 'Corn', 1: 'Cotton', 2: 'Rice', 3: 'Soybeans', 4: 'Others'}
CAL_LABELS = {0: 'Rice', 1: 'Alfalfa', 2: 'Grapes', 3: 'Almonds', 4: 'Pistachios', 5: 'Others'}

SAMPLES_PER_CLASS = 300
VAL_RATIO         = 0.2
RANDOM_SEED       = 42

os.makedirs(OUT_DIR, exist_ok=True)
print('Format cible : (N, C=10, T=36)  — channels-first')
print('Dossier de sortie :', OUT_DIR)
print('Index des bandes  :', IDX)

## Cellule 3 — Construction des 360 colonnes
Convention GEE : `t=0` → `B2..B12` (sans suffixe), `t=1` → `B2_1..B12_1`, ..., `t=35` → `B2_35..B12_35`

In [ ]:
def build_spectral_cols():
    cols = []
    for t in range(N_TIMES):
        for band in BANDS:
            cols.append(band if t == 0 else f'{band}_{t}')
    return cols

spectral_cols = build_spectral_cols()
print(f'Colonnes spectrales : {len(spectral_cols)}')
print(f'  t=0  : {spectral_cols[:10]}')
print(f'  t=1  : {spectral_cols[10:20]}')
print(f'  t=35 : {spectral_cols[-10:]}')

## Cellule 4 — Chargement & inspection des CSV

In [ ]:
ark_df = pd.read_csv(ARK_CSV)
cal_df = pd.read_csv(CAL_CSV)

for name, df, label_map in [('ARKANSAS', ark_df, ARK_LABELS), ('CALIFORNIA', cal_df, CAL_LABELS)]:
    print(f'=== {name} ===')
    print(f'  Shape : {df.shape}')
    for cls, count in df['label'].value_counts().sort_index().items():
        print(f'    Classe {int(cls)} ({label_map[int(cls)]:12s}) : {count:5d} ({count/len(df)*100:.1f}%)')
    print(f'  NaN : {df[spectral_cols].isna().sum().sum()}')
    missing = [c for c in spectral_cols if c not in df.columns]
    print(f'  Colonnes manquantes : {len(missing)}  {"✅" if not missing else "⚠️ "+str(missing[:3])}')
    print()

## Cellule 5 — Extraction brute → shape **(N, 10, 36)**

Le reshape se fait en deux étapes :
1. `(N, 360)` → `(N, 36, 10)` — ordre naturel des colonnes GEE
2. `.transpose(0, 2, 1)` → `(N, 10, 36)` — channels-first pour le modèle

In [ ]:
def extract_raw(df):
    """
    Extrait les features brutes au format (N, 10, 36) — channels-first.

    Étapes :
      1. (N, 360) → (N, 36, 10)  : reshape naturel (temps, bandes)
      2. (N, 36, 10) → (N, 10, 36) : transpose pour channels-first
    """
    X = df[spectral_cols].values.astype(np.float32)
    X = X.reshape(len(X), N_TIMES, N_BANDS)
    X = X.transpose(0, 2, 1)
    return X

ark_raw = extract_raw(ark_df)
cal_raw = extract_raw(cal_df)

print('Raw Arkansas  :', ark_raw.shape, '→ (N, bandes=10, temps=36)')
print('Raw California:', cal_raw.shape)
print()

val_check = ark_df[spectral_cols].values[0, 0]
print(f'Vérification cohérence : ark_raw[0, IDX[B2]=0, t=0] = {ark_raw[0, 0, 0]}')
print(f'  vs ark_df[B2][0]                                   = {val_check}  ✅' if ark_raw[0,0,0] == val_check else '  ❌ INCOHÉRENCE')

## Cellule 6 — Masque Input 2 → shape **(N, 36)**

Un pas de temps `t` est **manquant** si toutes ses 10 bandes valent 0.  
Avec le format `(N, 10, 36)` : les bandes sont en **axis=1** → `.all(axis=1)`

In [ ]:
def build_mask(X_raw):
    """
    Construit Input2 : shape (N, 36)
    1 = valide, 0 = manquant

    X_raw : (N, 10, 36) — bandes en axis=1
    → .all(axis=1) réduit la dimension bandes → (N, 36)
    """
    all_zero = (X_raw == 0).all(axis=1)
    mask = (~all_zero).astype(np.float32)
    return mask

ark_mask = build_mask(ark_raw)
cal_mask = build_mask(cal_raw)

for name, mask in [('Arkansas', ark_mask), ('California', cal_mask)]:
    miss_rate   = (mask == 0).mean() * 100
    miss_per_px = (mask == 0).sum(axis=1)
    print(f'=== {name} ===')
    print(f'  Shape masque           : {mask.shape}  (N, T=36)')
    print(f'  Taux global manquants  : {miss_rate:.2f}%')
    print(f'  Pixels sans manquant   : {(miss_per_px==0).sum():5d} ({(miss_per_px==0).mean()*100:.1f}%)')
    print(f'  Pixels >10 manquants   : {(miss_per_px>10).sum():5d}')
    print(f'  Valeurs uniques        : {np.unique(mask)}')
    print()

## Cellule 7 — Normalisation → `X / 10000`

Conforme au paper Section 2.2.4. Les valeurs manquantes (= 0) restent à 0.

In [ ]:
def normalize(X_raw):
    """
    Normalisation Sentinel-2 : division par 10000.
    Shape inchangée : (N, 10, 36) → (N, 10, 36)
    Valeurs manquantes (0 dans X_raw) restent à 0.
    """
    return (X_raw / SCALE).astype(np.float32)

ark_input1 = normalize(ark_raw)
cal_input1 = normalize(cal_raw)

for name, X in [('Arkansas', ark_input1), ('California', cal_input1)]:
    print(f'{name} — Input1 : {X.shape}  min={X.min():.4f}  max={X.max():.4f}  dtype={X.dtype}')

## Cellule 8 — Extraction des labels

In [ ]:
ark_labels = ark_df['label'].values.astype(np.int64)
cal_labels = cal_df['label'].values.astype(np.int64)

for region, labels, label_map in [
    ('ARKANSAS',   ark_labels, ARK_LABELS),
    ('CALIFORNIA', cal_labels, CAL_LABELS)
]:
    print(f'=== {region} ===')
    print(f'  Shape : {labels.shape}  dtype : {labels.dtype}')
    for cls, name in label_map.items():
        n = (labels == cls).sum()
        print(f'  Classe {cls} ({name:12s}) : {n:5d} ({n/len(labels)*100:.1f}%)')
    print()

## Cellule 9 — Vérification d'intégrité

Vérifications : shapes, NaN/Inf, valeurs masque, cohérence masque/Input1.  
**Format attendu : `(N, 10, 36)`**

In [ ]:
def check_integrity(input1, input2, labels, label_map, name):
    errors, warnings_list = [], []
    N = len(labels)

    if input1.shape != (N, N_BANDS, N_TIMES):
        errors.append(f'Input1 shape {input1.shape} ≠ attendu ({N}, {N_BANDS}, {N_TIMES})')
    if input2.shape != (N, N_TIMES):
        errors.append(f'Input2 shape {input2.shape} ≠ attendu ({N}, {N_TIMES})')
    if np.isnan(input1).any():
        errors.append(f'Input1 contient {np.isnan(input1).sum()} NaN')
    if np.isinf(input1).any():
        errors.append(f'Input1 contient {np.isinf(input1).sum()} Inf')
    if set(np.unique(input2)) - {0.0, 1.0}:
        errors.append(f'Input2 valeurs hors [0,1] : {np.unique(input2)}')
    if not set(np.unique(labels)).issubset(set(label_map.keys())):
        errors.append(f'Labels classes inconnues : {np.unique(labels)}')

    missing_2d = (input2 == 0)
    missing_3d = np.broadcast_to(
        missing_2d[:, np.newaxis, :], input1.shape
    )
    masked_vals = input1[missing_3d]
    n_nonzero = (masked_vals != 0).sum()
    if n_nonzero > 0:
        warnings_list.append(f'{n_nonzero} valeurs non-nulles aux positions masquées')

    print(f'=== {name} — VÉRIFICATION ===')
    if errors:
        for e in errors: print(f'  ❌ {e}')
    else:
        print(f'  ✅ Aucune erreur')
    if warnings_list:
        for w in warnings_list: print(f'  ⚠️  {w}')
    print(f'  Input1 : {input1.shape}  min={input1.min():.4f}  max={input1.max():.4f}')
    print(f'  Input2 : {input2.shape}  taux manquants={(input2==0).mean()*100:.2f}%')
    print(f'  Labels : {labels.shape}  classes={sorted(np.unique(labels).tolist())}')
    print()

check_integrity(ark_input1, ark_mask, ark_labels, ARK_LABELS, 'ARKANSAS')
check_integrity(cal_input1, cal_mask, cal_labels, CAL_LABELS, 'CALIFORNIA')

## Cellule 10 — Split train/val/test (conforme paper Table 2)

- 300 échantillons/classe → train+val (80/20)
- Reste → test

In [ ]:
def split_dataset(input1, input2, labels, label_map,
                  n_per_class=SAMPLES_PER_CLASS,
                  val_ratio=VAL_RATIO,
                  seed=RANDOM_SEED):
    """
    Split train/val/test conforme au paper (Table 2).
    input1 shape : (N, 10, 36) — fonctionne pour tout format (indexing sur axis=0)
    """
    np.random.seed(seed)
    train_idx, val_idx, test_idx = [], [], []

    for cls in sorted(label_map.keys()):
        idx_cls = np.where(labels == cls)[0]
        n_use   = min(n_per_class, len(idx_cls))
        np.random.shuffle(idx_cls)
        idx_trainval = idx_cls[:n_use]
        idx_test     = idx_cls[n_use:]
        idx_train, idx_val = train_test_split(
            idx_trainval, test_size=val_ratio, random_state=seed
        )
        train_idx.extend(idx_train)
        val_idx.extend(idx_val)
        test_idx.extend(idx_test)

    def subset(idx):
        idx = np.array(idx)
        return {'input1': input1[idx], 'input2': input2[idx], 'labels': labels[idx]}

    return subset(train_idx), subset(val_idx), subset(test_idx)

ark_train, ark_val, ark_test = split_dataset(ark_input1, ark_mask, ark_labels, ARK_LABELS)
cal_train, cal_val, cal_test = split_dataset(cal_input1, cal_mask, cal_labels, CAL_LABELS)

for name, train, val, test, label_map in [
    ('ARKANSAS',   ark_train, ark_val, ark_test, ARK_LABELS),
    ('CALIFORNIA', cal_train, cal_val, cal_test, CAL_LABELS)
]:
    print(f'=== {name} — Split ===')
    print(f'  Train  : {train["input1"].shape}  {len(train["labels"])} échantillons')
    print(f'  Val    : {val["input1"].shape}  {len(val["labels"])} échantillons')
    print(f'  Test   : {test["input1"].shape}  {len(test["labels"])} échantillons')
    print(f'  Total  : {len(train["labels"])+len(val["labels"])+len(test["labels"])}')
    print(f'  Train par classe :')
    for cls, cname in label_map.items():
        n = (train['labels'] == cls).sum()
        print(f'    {cls} ({cname:12s}) : {n}')
    print()

## Cellule 11 — Sauvegarde

In [ ]:
for region, input1, input2, labels, train, val, test in [
    ('Arkansas',   ark_input1, ark_mask, ark_labels, ark_train, ark_val, ark_test),
    ('California', cal_input1, cal_mask, cal_labels, cal_train, cal_val, cal_test),
]:

    np.save(os.path.join(OUT_DIR, f'{region}_input1.npy'), input1)
    np.save(os.path.join(OUT_DIR, f'{region}_input2.npy'), input2)
    np.save(os.path.join(OUT_DIR, f'{region}_labels.npy'), labels)

    for split_name, split in [('train', train), ('val', val), ('test', test)]:
        np.save(os.path.join(OUT_DIR, f'{region}_{split_name}_input1.npy'), split['input1'])
        np.save(os.path.join(OUT_DIR, f'{region}_{split_name}_input2.npy'), split['input2'])
        np.save(os.path.join(OUT_DIR, f'{region}_{split_name}_labels.npy'), split['labels'])

print('Fichiers sauvegardés :', OUT_DIR)
for f in sorted(os.listdir(OUT_DIR)):
    size = os.path.getsize(os.path.join(OUT_DIR, f)) / 1024
    print(f'  {f:48s} {size:8.1f} Ko')

## Cellule 12 — Vérification finale (rechargement)

In [ ]:
print('='*60)
for region, label_map in [('Arkansas', ARK_LABELS), ('California', CAL_LABELS)]:
    X   = np.load(os.path.join(OUT_DIR, f'{region}_input1.npy'))
    M   = np.load(os.path.join(OUT_DIR, f'{region}_input2.npy'))
    y   = np.load(os.path.join(OUT_DIR, f'{region}_labels.npy'))
    Xtr = np.load(os.path.join(OUT_DIR, f'{region}_train_input1.npy'))
    Xv  = np.load(os.path.join(OUT_DIR, f'{region}_val_input1.npy'))
    Xte = np.load(os.path.join(OUT_DIR, f'{region}_test_input1.npy'))

    print(f'\n  {region.upper()}')
    print(f'  Input1 : {X.shape}   ← (N, C=10, T=36) ✅')
    print(f'  Input2 : {M.shape}    ← (N, T=36) ✅')
    print(f'  Labels : {y.shape}     valeurs={sorted(np.unique(y).tolist())}')
    print(f'  min={X.min():.4f}  max={X.max():.4f}  NaN={np.isnan(X).sum()}')
    print(f'  Train  : {Xtr.shape}')
    print(f'  Val    : {Xv.shape}')
    print(f'  Test   : {Xte.shape}')
    print(f'  Total  : {len(Xtr)+len(Xv)+len(Xte)} / {len(y)}')

print()
print('✅  Preprocessing terminé — Format (N, 10, 36) prêt pour MCTNet !')

## Cellule 13 — Visualisation : taux de données manquantes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
for ax, M_data, title in zip(axes, [ark_mask, cal_mask], ['Arkansas', 'California']):
    miss_pct = (M_data == 0).mean(axis=0) * 100
    days = [t * 10 for t in range(N_TIMES)]
    colors = ['#E74C3C' if v > 50 else '#F39C12' if v > 20 else '#3498DB' for v in miss_pct]
    ax.bar(days, miss_pct, width=8, color=colors)
    ax.set_xlabel('DOY', fontsize=11)
    ax.set_ylabel('% pixels manquants', fontsize=11)
    ax.set_title(f'{title} — Taux manquants par pas de temps', fontsize=12, fontweight='bold')
    ax.set_xlim(-5, 365)
    ax.set_ylim(0, 105)
    ax.axhline(50, color='gray', linestyle='--', alpha=0.5)
    ax.grid(axis='y', alpha=0.3)
    taux = (M_data == 0).mean() * 100
    ax.text(0.98, 0.95, f'Global : {taux:.1f}%', transform=ax.transAxes,
            ha='right', va='top', fontsize=10, color='#C0392B',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#FDECEA'))
plt.tight_layout()
plt.savefig('missing_rates.png', dpi=150, bbox_inches='tight')
plt.show()

## Cellule 14 — Visualisation : profils NDVI (Figure 2 du paper)

NDVI **pour visualisation uniquement** — pas une feature du modèle.  
Format `(N, 10, 36)` → bande sur **axis=1** : `X_raw[:, IDX['B8'], :]`

In [ ]:
def compute_ndvi(X_raw, mask):
    """
    NDVI — visualisation uniquement, pas une feature du modèle.
    X_raw : (N, 10, 36) → bandes sur axis=1
    """
    eps = 1e-8
    NIR  = X_raw[:, IDX['B8'], :]
    Red  = X_raw[:, IDX['B4'], :]
    ndvi = (NIR - Red) / (NIR + Red + eps)
    return ndvi * mask

ark_ndvi = compute_ndvi(ark_raw, ark_mask)
cal_ndvi = compute_ndvi(cal_raw, cal_mask)

COLORS = ['#2ecc71', '#e74c3c', '#3498db', '#f39c12', '#9b59b6', '#1abc9c']
days   = np.array([t * 10 for t in range(N_TIMES)])
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, ndvi_data, labels_data, label_map, title in [
    (axes[0], ark_ndvi, ark_labels, ARK_LABELS, 'Arkansas'),
    (axes[1], cal_ndvi, cal_labels, CAL_LABELS, 'California')
]:
    for cls_id, cls_name in label_map.items():
        cls_mask  = (labels_data == cls_id)
        if cls_mask.sum() < 10: continue
        cls_ndvi  = ndvi_data[cls_mask]
        valid_cnt = (cls_ndvi != 0).sum(axis=0) + 1e-8
        mean_ndvi = cls_ndvi.sum(axis=0) / valid_cnt
        ax.plot(days, mean_ndvi, label=cls_name,
                color=COLORS[cls_id % len(COLORS)],
                marker='o', markersize=3, linewidth=1.8)
    ax.set_xlabel('DOY', fontsize=11)
    ax.set_ylabel('NDVI moyen', fontsize=11)
    ax.set_title(f'{title} — Profils NDVI (Figure 2 du paper)', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_xlim(-5, 365)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('ndvi_profiles.png', dpi=150, bbox_inches='tight')
plt.show()
print('⚠️  NDVI = visualisation uniquement — pas une feature du modèle')

## Cellule 15 — Visualisation : distributions des bandes normalisées

Format `(N, 10, 36)` → bande `i` sur **axis=1** : `ark_input1[:, i, :]`

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.suptitle('Distribution des bandes normalisées — Arkansas (pixels valides)',
             fontsize=13, fontweight='bold')

for i, (band_name, ax) in enumerate(zip(BANDS, axes.flatten())):
    band_data    = ark_input1[:, i, :]
    valid_pixels = band_data[ark_mask == 1]

    ax.hist(valid_pixels, bins=60, color='#3498DB', alpha=0.7, edgecolor='none')
    ax.set_title(band_name, fontweight='bold')
    ax.set_xlabel('Réflectance [0-1]', fontsize=8)
    ax.set_ylabel('Fréquence', fontsize=8)
    mean_v = valid_pixels.mean()
    ax.axvline(mean_v, color='red', linestyle='--', linewidth=1, label=f'μ={mean_v:.3f}')
    ax.legend(fontsize=7)
    ax.tick_params(labelsize=7)

plt.tight_layout()
plt.savefig('band_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## Cellule 16 — Résumé final

In [ ]:
print('='*62)
print('   RÉSUMÉ PREPROCESSING MCTNet — FORMAT (N, 10, 36)')
print('='*62)

for region, X, M, y, train, val, test, label_map in [
    ('Arkansas',   ark_input1, ark_mask, ark_labels, ark_train, ark_val, ark_test, ARK_LABELS),
    ('California', cal_input1, cal_mask, cal_labels, cal_train, cal_val, cal_test, CAL_LABELS)
]:
    print(f'\n  {region}')
    print(f'  ┌────────────────────────────────────────────────────')
    print(f'  │ Input1 : {X.shape}  ← (N, C=10 bandes, T=36 temps)')
    print(f'  │ Input2 : {M.shape}    ← (N, T=36) masque ALPE')
    print(f'  │ Labels : {y.shape}         ← (N,) entiers')
    print(f'  │ Manquants : {(M==0).mean()*100:.2f}%  |  range: [{X.min():.3f}, {X.max():.3f}]')
    print(f'  ├────────────────────────────────────────────────────')
    print(f'  │ Train : {train["input1"].shape}  ({len(train["labels"])} éch.)')
    print(f'  │ Val   : {val["input1"].shape}  ({len(val["labels"])} éch.)')
    print(f'  │ Test  : {test["input1"].shape}  ({len(test["labels"])} éch.)')
    print(f'  └────────────────────────────────────────────────────')

print()
print('  Normalisation : X / 10000  (paper Section 2.2.4)')
print('  NDVI/EVI/NDWI : visualisation seulement, hors features')
print()
print('  Règle des axes pour (N, 10, 36) :')
print('    build_mask       → .all(axis=1)          # réduire dim bandes')
print('    check_integrity  → broadcast [:, newaxis, :]  # (N,1,36)→(N,10,36)')
print('    compute_ndvi     → X[:, IDX["B8"], :]    # sélect. bande')
print('    band_dist        → X[:, i, :]             # sélect. bande i')
print()
print('✅  Prêt pour MCTNet !')